# Logistic Regression - Customer Churn Prediction 📊

## 🎯 Goal
Build a **customer churn predictor** - identify which customers are likely to cancel their subscription so the business can intervene with retention offers!

## 💰 Business Context
It costs **5× more** to acquire a new customer than retain an existing one. Predicting churn lets companies save millions by proactively offering discounts, upgrades, or personal outreach to at-risk customers.

## 📚 What You'll Learn
- The sigmoid function and probability
- How to train a churn prediction model
- How to interpret coefficients (negative vs positive)
- Decision boundaries
- All evaluation metrics (accuracy, precision, recall, F1, ROC-AUC)
- Why **RECALL** is critical for churn prediction
- How to tune the threshold for better business outcomes

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

np.random.seed(42)
sns.set_style('whitegrid')

print('✅ Libraries imported!')

## Step 2: Visualize the Sigmoid Function

The magic behind logistic regression - turns any number into a probability!

In [ ]:
def sigmoid(z):
    """The sigmoid function - converts any number to 0-1"""
    return 1 / (1 + np.exp(-z))

# Test sigmoid with different values
test_values = [-5, -2, -1, 0, 1, 2, 5]
print('Sigmoid Examples:')
print('=' * 40)
for z in test_values:
    print(f'  σ({z:3d}) = {sigmoid(z):.4f} ({sigmoid(z)*100:.1f}%)')

# Plot the sigmoid
z = np.linspace(-10, 10, 100)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(z, sigmoid(z), linewidth=3, color='blue')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold (0.5)')
ax.axvline(x=0, color='green', linestyle='--', alpha=0.5, label='z = 0')
ax.scatter([0], [0.5], color='red', s=100, zorder=5)
ax.set_xlabel('Input (z)', fontsize=12)
ax.set_ylabel('Output σ(z) = P(churn)', fontsize=12)
ax.set_title('The Sigmoid Function: Squishes Any Number to 0-1', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 3: Create Customer Dataset

Generate realistic customer data with features that influence churn:
- **tenure_months**: How long they've been a customer (1-36 months)
- **monthly_charges**: How much they pay per month ($20-$120)

**Intuition:** New customers with high bills are most likely to churn!

In [ ]:
# Generate customer data
n_customers = 400

# Features
tenure_months = np.random.uniform(1, 36, n_customers)
monthly_charges = np.random.uniform(20, 120, n_customers)

# True relationship (with noise):
# Short tenure + high charges → more likely to churn
score = 4 - 0.2*tenure_months + 0.04*monthly_charges + np.random.normal(0, 1, n_customers)
churned = (score > 0).astype(int)

df = pd.DataFrame({
    'tenure_months': tenure_months.round(1),
    'monthly_charges': monthly_charges.round(2),
    'churned': churned
})

print('📊 Customer Dataset:')
print(df.head(10))
print(f'\nTotal customers: {len(df)}')
print(f'Churned:  {df["churned"].sum()} ({df["churned"].mean()*100:.1f}%)')
print(f'Retained: {(1-df["churned"]).sum()} ({(1-df["churned"]).mean()*100:.1f}%)')

print('\n💡 Example customer profiles:')
print('  New customer, $100/mo → High churn risk 🚨')
print('  Loyal 2-year customer, $50/mo → Low churn risk ✅')

## Step 4: Visualize the Customer Data

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Plot retained customers
ax.scatter(df[df['churned']==0]['tenure_months'], df[df['churned']==0]['monthly_charges'],
           c='green', label='✅ Stayed (Loyal)', s=80, edgecolors='black', alpha=0.7)

# Plot churned customers
ax.scatter(df[df['churned']==1]['tenure_months'], df[df['churned']==1]['monthly_charges'],
           c='red', label='🚨 Churned', s=80, edgecolors='black', alpha=0.7, marker='X')

ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Monthly Charges ($)', fontsize=12)
ax.set_title('Customer Churn Analysis', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n💡 Notice the pattern:')
print('   Top-left (short tenure + high charges) = mostly churners 🚨')
print('   Bottom-right (long tenure + low charges) = mostly loyal ✅')

## Step 5: Prepare Data and Train Model

In [ ]:
# Split data
X = df[['tenure_months', 'monthly_charges']]
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features (good practice!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

print('🤖 Churn Predictor trained!')
print(f'\n📋 Model Parameters:')
print(f'  Intercept (β₀):       {model.intercept_[0]:.4f}')
print(f'  Tenure coef:          {model.coef_[0][0]:.4f}')
print(f'  Monthly charges coef: {model.coef_[0][1]:.4f}')

print('\n💡 Interpretation:')
print('  - Tenure coef is NEGATIVE → longer tenure = less churn (loyal!) ✅')
print('  - Charges coef is POSITIVE → higher bills = more churn ⚠️')

## Step 6: Make Predictions on Real Customers

In [ ]:
# Get predictions and probabilities on test set
predictions = model.predict(X_test_scaled)
probabilities = model.predict_proba(X_test_scaled)

# Show first 10 predictions
results_df = pd.DataFrame({
    'Tenure (mo)': X_test['tenure_months'].values[:10],
    'Monthly $': X_test['monthly_charges'].values[:10],
    'P(Stay)': probabilities[:10, 0].round(3),
    'P(Churn)': probabilities[:10, 1].round(3),
    'Predicted': ['🚨 Churn' if p == 1 else '✅ Stay' for p in predictions[:10]],
    'Actual': ['🚨 Churn' if a == 1 else '✅ Stay' for a in y_test.values[:10]]
})

results_df['Correct'] = results_df['Predicted'] == results_df['Actual']
print('🔮 First 10 Customer Predictions:')
print(results_df.to_string(index=False))

## Step 7: Evaluate the Churn Predictor

In [ ]:
# Calculate all metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

print('📊 Churn Predictor Performance:')
print('=' * 70)
print(f'  Accuracy:  {accuracy:.2%}')
print(f'  Precision: {precision:.2%}  ← Of flagged churners, how many really churn?')
print(f'  Recall:    {recall:.2%}  ← Of actual churners, how many did we catch?')
print(f'  F1 Score:  {f1:.2%}')

print('\n📋 Full Classification Report:')
print(classification_report(y_test, predictions, target_names=['Stayed', 'Churned']))

print('\n💡 For CHURN PREDICTION, RECALL is critical!')
print('   - Missing a churner = lost revenue forever 💸')
print('   - False alarm = just an unneeded retention offer (cheap!)')
print('   - Better to flag too many than miss real churners')

## Step 8: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['✅ Stayed', '🚨 Churned'],
            yticklabels=['✅ Stayed', '🚨 Churned'],
            cbar_kws={'label': 'Count'},
            ax=ax, annot_kws={'fontsize': 16})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Churn Predictor Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

TN, FP = cm[0]
FN, TP = cm[1]

print(f'\n📊 Business Impact Breakdown:')
print(f'  ✅ True Negatives:  {TN}  (correctly predicted loyal)')
print(f'  ⚠️ False Positives: {FP}  (loyal flagged as churn - wasted retention $)')
print(f'  ❌ False Negatives: {FN}  (churners we MISSED - lost revenue!)')
print(f'  ✅ True Positives:  {TP}  (churners we caught and can save!)')

if FN > 0:
    print(f'\n   💸 {FN} churners slipped through. Consider LOWER threshold.')

## Step 9: Visualize the Decision Boundary

See where the model separates likely churners from loyal customers!

In [ ]:
# Create mesh
x_min, x_max = X['tenure_months'].min() - 1, X['tenure_months'].max() + 1
y_min, y_max = X['monthly_charges'].min() - 5, X['monthly_charges'].max() + 5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# Scale and predict
grid = np.c_[xx.ravel(), yy.ravel()]
grid_scaled = scaler.transform(grid)
Z = model.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)

# Plot
fig, ax = plt.subplots(figsize=(12, 7))

# Probability contour (red = churn zone)
contour = ax.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.5)
plt.colorbar(contour, label='P(Churn)')

# Decision boundary at 0.5
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=3)

# Data points
ax.scatter(df[df['churned']==0]['tenure_months'], df[df['churned']==0]['monthly_charges'],
           c='green', label='✅ Stayed (Loyal)', s=80, edgecolors='black', alpha=0.8)
ax.scatter(df[df['churned']==1]['tenure_months'], df[df['churned']==1]['monthly_charges'],
           c='red', label='🚨 Churned', s=80, edgecolors='black', alpha=0.8, marker='X')

ax.set_xlabel('Tenure (months)', fontsize=12)
ax.set_ylabel('Monthly Charges ($)', fontsize=12)
ax.set_title('Customer Churn Decision Boundary', fontsize=14, fontweight='bold')
ax.legend(fontsize=12, loc='upper right')
plt.tight_layout()
plt.show()

print('\n💡 The black line is the decision boundary (P=0.5)')
print('   Upper-left of line → Likely to churn | Lower-right → Loyal')
print('   Color shows confidence (redder = higher churn risk)')

## Step 10: ROC Curve and AUC

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, probabilities[:, 1])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, color='blue', linewidth=3, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.2, color='blue')
ax.set_xlabel('False Positive Rate (loyal flagged as churn)', fontsize=12)
ax.set_ylabel('True Positive Rate (churners caught)', fontsize=12)
ax.set_title('Churn Predictor ROC Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n📊 AUC Score: {roc_auc:.3f}')
if roc_auc >= 0.9:
    print('   ✅ Excellent churn predictor!')
elif roc_auc >= 0.8:
    print('   ✅ Good churn predictor!')
elif roc_auc >= 0.7:
    print('   ✅ Decent churn predictor')
else:
    print('   ⚠️ Needs improvement')

## Step 11: Tune Threshold for Better Business Outcomes

**For churn, we want HIGH RECALL** - catch as many at-risk customers as possible!
Let's try LOWER thresholds.

In [ ]:
thresholds_to_try = [0.2, 0.3, 0.5, 0.7]
results = []

for thresh in thresholds_to_try:
    pred_thresh = (probabilities[:, 1] >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, pred_thresh)
    fn = cm_t[1, 0]  # Churners we missed
    
    results.append({
        'Threshold': thresh,
        'Precision': precision_score(y_test, pred_thresh, zero_division=0),
        'Recall': recall_score(y_test, pred_thresh, zero_division=0),
        'F1': f1_score(y_test, pred_thresh, zero_division=0),
        'Missed Churners': fn,
        'Business Impact': '💸 Bad' if fn > 5 else '✅ Good'
    })

thresh_df = pd.DataFrame(results)
print('🎚️ Threshold Tuning for Churn Prediction:')
print('=' * 80)
print(thresh_df.round(3).to_string(index=False))

print('\n💡 Key insight for churn prediction:')
print('   - LOWER threshold (0.3) → catches more churners (higher recall) ✅')
print('   - HIGHER threshold (0.7) → fewer false alarms but misses churners 💸')
print('   - Best for business: 0.3-0.4 to catch maximum at-risk customers!')

## Step 12: Build a Churn Risk Predictor!

A function to score any customer and recommend a business action.

In [ ]:
def predict_churn(tenure_months, monthly_charges, customer_name='Customer'):
    """Predict churn risk and recommend business action"""
    features = np.array([[tenure_months, monthly_charges]])
    features_scaled = scaler.transform(features)
    
    prob = model.predict_proba(features_scaled)[0, 1]
    
    print(f'\n👤 {customer_name}:')
    print(f'   Tenure: {tenure_months} months')
    print(f'   Monthly bill: ${monthly_charges}')
    print(f'\n   📊 P(Churn) = {prob:.1%}')
    
    if prob >= 0.8:
        print(f'   🚨 VERY HIGH RISK')
        print(f'   💡 Action: Personal call from account manager + 20% discount')
    elif prob >= 0.6:
        print(f'   ⚠️ HIGH RISK')
        print(f'   💡 Action: Email retention offer + free upgrade')
    elif prob >= 0.3:
        print(f'   🟡 MODERATE RISK')
        print(f'   💡 Action: Send loyalty email, monitor closely')
    else:
        print(f'   ✅ LOW RISK - Loyal customer')
        print(f'   💡 Action: No intervention needed')

# Test with various customer profiles
predict_churn(tenure_months=2, monthly_charges=110, customer_name='Alice (new, high bill)')
predict_churn(tenure_months=6, monthly_charges=85, customer_name='Bob (6 months, medium bill)')
predict_churn(tenure_months=18, monthly_charges=70, customer_name='Carol (1.5 years)')
predict_churn(tenure_months=30, monthly_charges=50, customer_name='Dave (2.5 years, cheap plan)')

print('\n\n💼 Business ROI Example:')
print('   Say retention offer costs $20 and each saved customer = $500 LTV')
print('   Even if offer only works 20% of time: 0.2 × $500 - $20 = $80 profit per flagged customer!')
print('   → Worth targeting anyone with P(churn) > 10% in real business!')

## 🎓 Summary

### What We Built:
1. ✅ Visualized the sigmoid function
2. ✅ Created realistic customer dataset
3. ✅ Trained a Logistic Regression churn predictor
4. ✅ Made predictions with confidence scores
5. ✅ Evaluated with multiple metrics
6. ✅ Visualized confusion matrix and decision boundary
7. ✅ Plotted ROC curve and calculated AUC
8. ✅ Tuned threshold to maximize churn detection
9. ✅ Built a churn risk predictor with business actions!

### Key Takeaways:
- **Sigmoid** converts any number to a probability (0-1)
- **Negative coefficients** mean the feature DECREASES the outcome
- **RECALL matters most** for churn prediction (don't miss at-risk customers!)
- **Lower threshold** (0.3) is better than 0.5 for churn
- **Decision boundary** shows where model is uncertain

### Real-World Churn Prediction Considerations:
1. **Many more features**: contract type, payment method, support tickets, usage patterns, demographics
2. **Imbalanced data**: typically only 10-20% churn rate
3. **Time-based features**: seasonality, day-of-week signups
4. **Cost-sensitive training**: false negatives cost MORE than false positives
5. **Ensemble methods** (Random Forest, XGBoost) often perform better

### Try These Exercises:
1. Add more features (contract_type, num_support_calls, payment_method)
2. Try with imbalanced data (class_weight='balanced')
3. Add regularization (penalty='l1' or 'l2')
4. Use the real **Telco Customer Churn** dataset from Kaggle
5. Calculate actual ROI based on retention cost vs customer LTV

**Happy Customer Retention! 📊🎯✨**